## Removing usless cells or unused cells that mean nothing


In [7]:
!pip install ultralytics # installing ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 34.4 MB/s eta 0:00:00


In [ ]:
# mounting Drive
from google.colab import drive
drive.mount('/content/drive',force_remount=True)

# testing the drive in the same cell
!ls /content/drive/MyDrive/GP/YoloModels

Mounted at /content/drive
'Copy of YOLO26_run.ipynb'   tuned_Yolo   YOLO+landmark
 RetrainedYolo8		     Yolo26	 'نسخة من Copy gh of YOLO26_run.ipynb'
 toStream		     Yolo5


In [ ]:
from ultralytics import YOLO

model = YOLO("/content/drive/MyDrive/GP/YoloModels/Yolo26/best.pt")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
import os
import numpy as np

def process_test_sequence_fixed(folder, det_conf=0.25, closed_thr=0.5, open_thr=0.5,
                                min_blink_frames=3, gap_merge=2):
    files = sorted([f for f in os.listdir(folder) if f.endswith(('.jpg', '.png', '.jpeg'))]) # sort files becasue we need them sorted

    state_seq = []
    closed_conf_seq = [] # i feel like we dont need it

    prev_state = 0  # assume open at start fall back

    for f in files:
        img_path = os.path.join(folder, f)
        res = model.predict(img_path, conf=det_conf, verbose=False)

        closed_conf = 0.0
        open_conf = 0.0


        if res and len(res) > 0 and res[0].boxes is not None and len(res[0].boxes) > 0:
            for box in res[0].boxes:
                conf = float(box.conf[0])
                cls_id = int(box.cls[0])
                class_name = model.names[cls_id]

                if class_name == 'closed eyes':
                    closed_conf = max(closed_conf, conf)
                elif class_name == 'opened eyes':
                    open_conf = max(open_conf, conf)

        closed_conf_seq.append(closed_conf)
        print(f"\nImage: {f}")
        print(f"Closed conf: {closed_conf}, Open conf: {open_conf:.4f}")

        # Stable frame-state decision
        if closed_conf >= closed_thr :# closed_conf >= open_conf cause model is bias towerd open
            state = 1
            print("inside closed")
            print(f"Closed conf: {closed_conf}")
        elif open_conf >= open_thr and open_conf > closed_conf:
            state = 0
            print("inside open")
            print(f"Closed conf: {closed_conf}")
        else:
            state = 0 # carry previous state if uncertain / no detection

        state_seq.append(state)
        prev_state = state

    state_seq = np.array(state_seq, dtype=int)
    closed_conf_seq = np.array(closed_conf_seq, dtype=float)
    print("closed conf min/max:", min(closed_conf_seq), max(closed_conf_seq))
    print("first 50 closed confs:", closed_conf_seq[:50])#good
    # Remove tiny glitches: 0,1,0 or 1,0,1
    for i in range(1, len(state_seq) - 1):
        if state_seq[i - 1] == state_seq[i + 1]:
            state_seq[i] = state_seq[i - 1]

    # Extract closed runs
    raw_blinks = []
    in_blink = False

    si = -1
    print("num closed frames:", int(np.sum(state_seq)))

    for t in range(len(state_seq)):
        if not in_blink and state_seq[t] == 1:
            si = t
            in_blink = True
        elif in_blink and state_seq[t] == 0:
            ei = t - 1
            if ei >= si:
                raw_blinks.append((si, ei))
            in_blink = False #set them again
            si = -1

    if in_blink and si != -1:
        raw_blinks.append((si, len(state_seq) - 1))

    # Merge blink fragments separated by tiny open gaps
    merged = []
    for seg in raw_blinks:
        if not merged:
            merged.append(seg)
        else:
            prev_si, prev_ei = merged[-1]
            cur_si, cur_ei = seg
            if cur_si - prev_ei - 1 <= gap_merge:
                merged[-1] = (prev_si, cur_ei)
            else:
                merged.append(seg)

    print("Raw closed runs:")
    for si, ei in merged:
      print((si, ei), "duration =", ei - si + 1)


    # Keep only realistic blink durations AFTER trimming
    blinks = []
    for si, ei in merged:
        duration = ei - si + 1
        if min_blink_frames <= duration :
            blinks.append((si, ei))

    print("State sequence:")
    print(state_seq.tolist())
    print("\nDetected blinks:", blinks)

    if not blinks:
        print("No blink detected.")
        return

    maxi = 0
    for blink_idx, (si, ei) in enumerate(blinks):
        duration = ei - si + 1
        maxi = max(maxi, duration)

        # peak inside blink from closed confidence
        seg = closed_conf_seq[si:ei+1]
        bi = si + int(np.argmax(seg))

        baseline = min(closed_conf_seq[si], closed_conf_seq[ei])
        amplitude = closed_conf_seq[bi] - baseline

        if ei > bi:
            velocity = (closed_conf_seq[bi] - closed_conf_seq[ei]) / (ei - bi)
        else:
            velocity = 0.0#to avoid runtime error

        frequency = 100 * ((blink_idx + 1) / (ei + 1))#consider changing it
        print("start value:", closed_conf_seq[si])
        print("end value:", closed_conf_seq[ei])
        print(f"\n--- Blink {blink_idx + 1} ---")
        print(f"Interval B_i: [{si}, {ei}]")
        print(f"Duration: {duration} frames")
        print(f"baseline: {baseline} ")
        print(f"peak: {bi} ")
        print(f"Amplitude: {amplitude:.4f}")
        print(f"Reopening Velocity: {velocity:.4f}")
        print(f"Blink Frequency: {frequency:.2f} per 100 frames")
        print("first 50 closed confs:", closed_conf_seq[:50])
    print(f"\nThe maximum duration of a blink is {maxi}")
    for i in range(len(closed_conf_seq)):
     print(i, closed_conf_seq[i])


##Trying to make the yolo detection faster

#baseline

In [ ]:
# Can be used for perclose only and for the logistic regrission
import numpy as np

def compute_perclos(state_seq, window_size=150, threshold=0.8):
    """
    state_seq: array/list where 1 = closed eyes, 0 = open eyes
    window_size: number of frames in each temporal window
                 example: 150 frames = 5 seconds if FPS = 30
    threshold: drowsiness threshold
    """
    state_seq = np.array(state_seq)

    perclos_values = []
    drowsy_predictions = []

    for t in range(window_size - 1, len(state_seq)):
        window = state_seq[t - window_size + 1 : t + 1]

        perclos = np.mean(window)
        drowsy = 1 if perclos > threshold else 0

        perclos_values.append(perclos)
        drowsy_predictions.append(drowsy)

    return np.array(perclos_values), np.array(drowsy_predictions)

In [ ]:
import torch, gc

gc.collect()
torch.cuda.empty_cache()

In [ ]:
!nvidia-smi

Thu May  7 19:56:06 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   35C    P0             58W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()

In [ ]:
def extract_blinks_from_state_seq(state_seq, min_blink_frames=3):
    """
    Finds closed-eye events from binary sequence.
    """
    blinks = []
    start = None

    for i, state in enumerate(state_seq):
        if state == 1 and start is None:
            start = i

        elif state == 0 and start is not None:
            end = i - 1
            duration = end - start + 1

            if duration >= min_blink_frames:
                blinks.append((start, end, duration))

            start = None

    if start is not None:
        end = len(state_seq) - 1
        duration = end - start + 1

        if duration >= min_blink_frames:
            blinks.append((start, end, duration))

    return blinks

In [ ]:
def extract_window_features(state_seq, fps=30, window_size=150, step_size=30, min_blink_frames=3):
    state_seq = np.array(state_seq)
    features = []

    for start in range(0, len(state_seq) - window_size + 1, step_size):
        end = start + window_size
        window = state_seq[start:end]

        perclos = np.mean(window)

        blinks = extract_blinks_from_state_seq(
            window,
            min_blink_frames=min_blink_frames
        )

        if len(blinks) > 0:
            durations_frames = [b[2] for b in blinks]
            mean_blink_duration = np.mean(durations_frames) / fps
        else:
            mean_blink_duration = 0

        window_seconds = window_size / fps
        blink_rate = len(blinks) / window_seconds

        features.append([
            perclos,
            mean_blink_duration,
            blink_rate
        ])

    return np.array(features)

In [ ]:
import os, gc
import numpy as np
import torch
from PIL import Image, UnidentifiedImageError

def is_valid_image(path):
    try:
        with Image.open(path) as img:
            img.verify()
        return True
    except Exception:
        return False


def process_video_folder_safe(
    folder,
    det_conf=0.1,
    closed_thr=0.25,
    open_thr=0.25,
    imgsz=320,
    device=0,
    chunk_size=100
):
    files = sorted([
        f for f in os.listdir(folder)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ])

    state_seq = []
    closed_conf_seq = []

    prev_state = 0

    for i in range(0, len(files), chunk_size):
        chunk_files = files[i:i + chunk_size]

        image_paths = []
        valid_files = []

        for f in chunk_files:
            path = os.path.join(folder, f)

            if is_valid_image(path):
                image_paths.append(path)
                valid_files.append(f)
            else:
                print("Skipping corrupted image:", path)

        if len(image_paths) == 0:
            continue

        results_generator = model.predict(
            source=image_paths,
            conf=det_conf,
            imgsz=imgsz,
            stream=True,
            verbose=False,
            device=device
        )

        for res in results_generator:
            closed_conf = 0.0
            open_conf = 0.0

            if res.boxes is not None and len(res.boxes) > 0:
                for box in res.boxes:
                    conf = float(box.conf[0])
                    cls_id = int(box.cls[0])
                    class_name = model.names[cls_id]

                    if class_name == "closed eyes":
                        closed_conf = max(closed_conf, conf)
                    elif class_name == "opened eyes":
                        open_conf = max(open_conf, conf)

            if closed_conf >= closed_thr and closed_conf >= open_conf:
                state = 1
            elif open_conf >= open_thr and open_conf > closed_conf:
                state = 0
            else:
                state = prev_state

            state_seq.append(state)
            closed_conf_seq.append(closed_conf)
            prev_state = state

            del res

        del image_paths, valid_files, results_generator
        gc.collect()
        torch.cuda.empty_cache()

    return {
        "state_seq": np.array(state_seq, dtype=np.int8),
        "closed_conf_seq": np.array(closed_conf_seq, dtype=np.float32)
    }

In [ ]:
import os

alert_root = "/content/drive/MyDrive/GP2/fps30_all_frames/fps30_traningFrames/Alert"
drowsy_root = "/content/drive/MyDrive/GP2/fps30_all_frames/fps30_traningFrames/Drowsy"

videos = []

# Alert videos label = 0
for name in sorted(os.listdir(alert_root)):
    path = os.path.join(alert_root, name)
    if os.path.isdir(path):
        videos.append((path, 0))

# Drowsy videos label = 1
for name in sorted(os.listdir(drowsy_root)):
    path = os.path.join(drowsy_root, name)
    if os.path.isdir(path):
        videos.append((path, 1))

print("Total videos:", len(videos))
print("Alert videos:", sum(label == 0 for _, label in videos))
print("Drowsy videos:", sum(label == 1 for _, label in videos))

print(videos[:5])

Total videos: 34
Alert videos: 20
Drowsy videos: 14
[('/content/drive/MyDrive/GP2/fps30_all_frames/fps30_traningFrames/Alert/A001_20260329_104639_frames', 0), ('/content/drive/MyDrive/GP2/fps30_all_frames/fps30_traningFrames/Alert/A003_20260329_105445_frames', 0), ('/content/drive/MyDrive/GP2/fps30_all_frames/fps30_traningFrames/Alert/A004_20260413_152711_frames', 0), ('/content/drive/MyDrive/GP2/fps30_all_frames/fps30_traningFrames/Alert/A005_20260413_154752_frames', 0), ('/content/drive/MyDrive/GP2/fps30_all_frames/fps30_traningFrames/Alert/A006_20260413_154904_frames', 0)]


In [ ]:
import os
import gc
import torch
import numpy as np

from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix



# 1) SAFE CHUNKED YOLO PROCESSING


def process_test_sequence_chunked(
    folder,
    det_conf=0.1,
    closed_thr=0.25,
    open_thr=0.25,
    imgsz=320,
    batch_size=4,
    chunk_size=200,
    device=0
):
    files = sorted([
        f for f in os.listdir(folder)
        if f.lower().endswith((".jpg", ".png", ".jpeg"))
    ])

    if len(files) == 0:
        print("No images found.")
        return None

    state_seq = []
    closed_conf_seq = []

    for start in range(0, len(files), chunk_size):
        end = min(start + chunk_size, len(files))
        chunk_files = files[start:end]

        print(f"Processing frames {start} to {end - 1}")

        valid_paths = []
        valid_files = []

        for f in chunk_files:
            path = os.path.join(folder, f)

            try:
                with Image.open(path) as img:
                    img.verify()

                valid_paths.append(path)
                valid_files.append(f)

            except Exception:
                print(f"Skipping corrupted image: {f}")

        if len(valid_paths) == 0:
            continue

        results = model.predict(
            valid_paths,
            conf=det_conf,
            imgsz=imgsz,
            batch=batch_size,
            device=device,
            verbose=False,
            save=False
        )

        for f, res in zip(valid_files, results):
            closed_conf = 0.0
            open_conf = 0.0

            if res.boxes is not None and len(res.boxes) > 0:
                for box in res.boxes:
                    conf = float(box.conf[0])
                    cls_id = int(box.cls[0])
                    class_name = model.names[cls_id]

                    if class_name == "closed eyes":
                        closed_conf = max(closed_conf, conf)

                    elif class_name == "opened eyes":
                        open_conf = max(open_conf, conf)

            closed_conf_seq.append(closed_conf)

            if closed_conf >= closed_thr:
                state = 1
            elif open_conf >= open_thr and open_conf > closed_conf:
                state = 0
            else:
                state = 0

            state_seq.append(state)

        del results, valid_paths, valid_files
        gc.collect()
        torch.cuda.empty_cache()

    return {
        "files": files,
        "state_seq": np.array(state_seq, dtype=int),
        "closed_conf_seq": np.array(closed_conf_seq, dtype=float)
    }


    #2) Extracting window features

    def extract_window_features(
    state_seq,
    fps=30,
    window_size=150,
    step_size=30,
    min_blink_frames=3
):
      state_seq = np.asarray(state_seq)
      features = []

      for start in range(0, len(state_seq) - window_size + 1, step_size):
        end = start + window_size
        window = state_seq[start:end]

        # PERCLOS
        perclos = np.mean(window)

        # Blink extraction inside this window
        blinks = []
        in_blink = False
        si = -1

        for i, state in enumerate(window):
            if state == 1 and not in_blink:
                si = i
                in_blink = True

            elif state == 0 and in_blink:
                ei = i - 1
                duration = ei - si + 1

                if duration >= min_blink_frames:
                    blinks.append((si, ei, duration))

                in_blink = False
                si = -1

        if in_blink:
            ei = len(window) - 1
            duration = ei - si + 1

            if duration >= min_blink_frames:
                blinks.append((si, ei, duration))

        blink_count = len(blinks)

        if blink_count > 0:
            durations = [b[2] for b in blinks]
            avg_blink_duration = np.mean(durations) / fps
            max_blink_duration = np.max(durations) / fps
        else:
            avg_blink_duration = 0.0
            max_blink_duration = 0.0

        window_seconds = window_size / fps
        blink_rate = blink_count / window_seconds

        features.append([
            perclos,
            blink_count,
            avg_blink_duration,
            max_blink_duration,
            blink_rate
        ])

    return np.array(features)


In [ ]:
######DO NOT PRESSS TAKES 6 Hours

# 3) BUILD DATASET

all_X = []
all_y = []
all_groups = []

for video_id, (folder, video_label) in enumerate(videos):

    print(f"\nProcessing video {video_id + 1}/{len(videos)}")
    print("Folder:", folder)
    print("Label:", "Drowsy" if video_label == 1 else "Alert")

    output = process_test_sequence_chunked(
        folder,
        det_conf=0.1,
        closed_thr=0.25,
        open_thr=0.25,
        imgsz=320,
        batch_size=4,
        chunk_size=200,
        device=0
    )

    if output is None:
        print("Skipped: no output")
        continue

    state_seq = output["state_seq"]

    X_video = extract_window_features(
        state_seq,
        fps=30,
        window_size=150,
        step_size=30,
        min_blink_frames=3
    )

    X_video = np.asarray(X_video)

    if X_video.ndim == 1:
        X_video = X_video.reshape(1, -1)

    if X_video.shape[0] == 0:
        print("Skipped: video too short for window extraction")
        del output, state_seq, X_video
        gc.collect()
        torch.cuda.empty_cache()
        continue

    y_video = np.full(X_video.shape[0], video_label)
    groups_video = np.full(X_video.shape[0], video_id)

    all_X.append(X_video)
    all_y.append(y_video)
    all_groups.append(groups_video)

    print("Windows extracted:", X_video.shape[0])

    del output, state_seq, X_video, y_video, groups_video
    gc.collect()
    torch.cuda.empty_cache()



# 4) COMBINE ALL VIDEOS


X = np.vstack(all_X)
y = np.concatenate(all_y)
groups = np.concatenate(all_groups)

print("\nFinal check:")
print("X shape:", X.shape)
print("y length:", len(y))
print("groups length:", len(groups))
print("Labels:", np.unique(y, return_counts=True))



# 5) CHECK CLASSES


if len(np.unique(y)) < 2:
    raise ValueError("You need both Alert videos and Drowsy videos to train logistic regression.")



Processing video 1/34
Folder: /content/drive/MyDrive/GP2/fps30_all_frames/fps30_traningFrames/Alert/A001_20260329_104639_frames
Label: Alert
Processing frames 0 to 199
Processing frames 200 to 399
Processing frames 400 to 599
Processing frames 600 to 799
Processing frames 800 to 999
Processing frames 1000 to 1199
Processing frames 1200 to 1399
Processing frames 1400 to 1599
Processing frames 1600 to 1799
Processing frames 1800 to 1999
Processing frames 2000 to 2199
Processing frames 2200 to 2399
Processing frames 2400 to 2599
Processing frames 2600 to 2799
Processing frames 2800 to 2999
Processing frames 3000 to 3199
Processing frames 3200 to 3399
Processing frames 3400 to 3599
Processing frames 3600 to 3799
Processing frames 3800 to 3999
Processing frames 4000 to 4199
Processing frames 4200 to 4399
Processing frames 4400 to 4599
Processing frames 4600 to 4799
Processing frames 4800 to 4999
Processing frames 5000 to 5199
Processing frames 5200 to 5399
Processing frames 5400 to 5599
Pr

In [ ]:
# defies the cheakpoint to not go back
import os
import pickle

CHECKPOINT_PATH = "/content/drive/MyDrive/GP2/checkpoints/progress.pkl"
os.makedirs(os.path.dirname(CHECKPOINT_PATH), exist_ok=True)

def save_checkpoint(data):
    with open(CHECKPOINT_PATH, "wb") as f:
        pickle.dump(data, f)
    print("Checkpoint saved.")

def load_checkpoint():
    if os.path.exists(CHECKPOINT_PATH):
        with open(CHECKPOINT_PATH, "rb") as f:
            print("Checkpoint loaded.")
            return pickle.load(f)
    return None

In [ ]:
# The train dataset
import pandas as pd
df_train = pd.read_csv("/content/drive/MyDrive/GP2/drowsiness_train_dataset.csv")

X = df_train[[
    "perclos",
    "blink_count",
    "avg_blink_duration",
    "max_blink_duration",
    "blink_rate"
]].values

y = df_train["label"].values
groups = df_train["video_id"].values


In [ ]:
import pandas as pd
df_train = pd.DataFrame(X, columns=[
    "perclos",
    "blink_count",
    "avg_blink_duration",
    "max_blink_duration",
    "blink_rate"
])
df_train["label"] = y
df_train["video_id"] = groups

save_path = "/content/drive/MyDrive/GP2/drowsiness_train_dataset.csv"

df_train.to_csv(save_path, index=False)

print("Training dataset saved to:")
print(save_path)
print("Dataset shape:", df_train.shape)
print("Labels:", df_train["label"].value_counts().sort_index().to_dict())

Training dataset saved to:
/content/drive/MyDrive/GP2/drowsiness_train_dataset.csv
Dataset shape: (6504, 7)
Labels: {0: 3662, 1: 2842}


In [ ]:
# Dont Presssss Test Takes a lonnnnnnnng time
# TEST ON SEPARATE TEST FOLDER WITH CHECKPOINT


import os
import pickle
import numpy as np

from google.colab import drive
drive.mount('/content/drive')

CHECKPOINT_PATH = "/content/drive/MyDrive/GP2/checkpoints/test_feature_checkpoint.pkl"
os.makedirs(os.path.dirname(CHECKPOINT_PATH), exist_ok=True)

def save_checkpoint(data):
    with open(CHECKPOINT_PATH, "wb") as f:
        pickle.dump(data, f)
    print("✅ Checkpoint saved")

def load_checkpoint():
    if os.path.exists(CHECKPOINT_PATH):
        with open(CHECKPOINT_PATH, "rb") as f:
            print("✅ Checkpoint loaded")
            return pickle.load(f)
    return None


test_alert_root = "/content/drive/MyDrive/GP2/fps30_all_frames/fps30_testFrames/Alert"
test_drowsy_root = "/content/drive/MyDrive/GP2/fps30_all_frames/fps30_testFrames/Drowsy"

test_videos = []

for folder_name in sorted(os.listdir(test_alert_root)):
    path = os.path.join(test_alert_root, folder_name)
    if os.path.isdir(path):
        test_videos.append((path, 0))

for folder_name in sorted(os.listdir(test_drowsy_root)):
    path = os.path.join(test_drowsy_root, folder_name)
    if os.path.isdir(path):
        test_videos.append((path, 1))

print("Total test videos:", len(test_videos))
print("Alert test videos:", sum(label == 0 for _, label in test_videos))
print("Drowsy test videos:", sum(label == 1 for _, label in test_videos))


# ==============================
# LOAD PREVIOUS PROGRESS
# ==============================

checkpoint = load_checkpoint()

if checkpoint is not None:
    test_X = checkpoint["test_X"]
    test_y = checkpoint["test_y"]
    test_groups = checkpoint["test_groups"]
    start_video = checkpoint["next_video"]
else:
    test_X = []
    test_y = []
    test_groups = []
    start_video = 0

print("Starting from video:", start_video + 1)


# ==============================
# EXTRACT TEST FEATURES
# ==============================

for video_id, (folder, video_label) in enumerate(test_videos[start_video:], start=start_video):

    print(f"\nProcessing test video {video_id + 1}/{len(test_videos)}")
    print("Folder:", folder)
    print("Label:", "Drowsy" if video_label == 1 else "Alert")

    output = process_test_sequence_chunked(
        folder,
        det_conf=0.1,
        closed_thr=0.25,
        open_thr=0.25,
        imgsz=320,
        batch_size=4,
        chunk_size=200,
        device=0
    )

    if output is None:
        print("Skipped: no output")

        save_checkpoint({
            "test_X": test_X,
            "test_y": test_y,
            "test_groups": test_groups,
            "next_video": video_id + 1
        })

        continue

    state_seq = output["state_seq"]

    X_video = extract_window_features(
        state_seq,
        fps=30,
        window_size=150,
        step_size=30,
        min_blink_frames=3
    )

    X_video = np.asarray(X_video)

    if X_video.ndim == 1:
        X_video = X_video.reshape(1, -1)

    if X_video.shape[0] == 0:
        print("Skipped: video too short for window extraction")

        save_checkpoint({
            "test_X": test_X,
            "test_y": test_y,
            "test_groups": test_groups,
            "next_video": video_id + 1
        })

        continue

    y_video = np.full(X_video.shape[0], video_label)
    groups_video = np.full(X_video.shape[0], video_id)

    test_X.append(X_video)
    test_y.append(y_video)
    test_groups.append(groups_video)

    print("Windows extracted:", X_video.shape[0])

    save_checkpoint({
        "test_X": test_X,
        "test_y": test_y,
        "test_groups": test_groups,
        "next_video": video_id + 1
    })



# COMBINE TEST DATA

if len(test_X) == 0:
    raise ValueError("No test features were extracted.")

X_test = np.vstack(test_X)
y_test = np.concatenate(test_y)
test_groups = np.concatenate(test_groups)

print("\nFinal test check:")
print("X_test shape:", X_test.shape)
print("y_test length:", len(y_test))
print("test groups length:", len(test_groups))
print("Test labels:", np.unique(y_test, return_counts=True))



# EVALUATE TRAINED MODEL


y_pred = clf.predict(X_test)

print("\n==============================")
print("SEPARATE TEST SET RESULTS")
print("==============================")

print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["Alert", "Drowsy"]))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Total test videos: 10
Alert test videos: 5
Drowsy test videos: 5
✅ Checkpoint loaded
Starting from video: 11

Final test check:
X_test shape: (2249, 3)
y_test length: 2249
test groups length: 2249
Test labels: (array([0, 1]), array([1206, 1043]))


ValueError: X has 3 features, but StandardScaler is expecting 5 features as input.

In [ ]:
# Test

import pandas as pd

df_test = pd.DataFrame(X_test, columns=[
    "perclos",
    "blink_count",
    "avg_blink_duration",
    "max_blink_duration",
    "blink_rate"
])

df_test["label"] = y_test
df_test["video_id"] = groups_test

test_save_path = "/content/drive/MyDrive/GP2/drowsiness_test_dataset.csv"

df_test.to_csv(test_save_path, index=False)

print("\nTest dataset saved to:")
print(test_save_path)
print("Dataset shape:", df_test.shape)
print("Labels:", df_test["label"].value_counts().sort_index().to_dict())

ValueError: Shape of passed values is (2249, 3), indices imply (2249, 5)

In [ ]:
import os
import pickle
import joblib
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


# PATHS


TRAIN_CSV_PATH = "/content/drive/MyDrive/GP2/drowsiness_train_dataset.csv"
TEST_CHECKPOINT_PATH = "/content/drive/MyDrive/GP2/checkpoints/test_feature_checkpoint.pkl"
MODEL_PATH = "/content/drive/MyDrive/GP2/checkpoints/clf_model_3features.pkl"


# LOAD TRAIN DATA


df_train = pd.read_csv(TRAIN_CSV_PATH)

# ONLY THE 3 FEATURES YOU WANT
X_train = df_train[[
    "perclos",
    "avg_blink_duration",
    "blink_rate"
]].values

y_train = df_train["label"].values

print("X_train shape:", X_train.shape)


# LOAD TEST CHECKPOINT


with open(TEST_CHECKPOINT_PATH, "rb") as f:
    checkpoint = pickle.load(f)

X_test = np.vstack(checkpoint["test_X"])
y_test = np.concatenate(checkpoint["test_y"])

print("X_test shape:", X_test.shape)


# TRAIN OR LOAD MODEL


if os.path.exists(MODEL_PATH):

    print("Loading saved model...")
    clf = joblib.load(MODEL_PATH)

else:

    print("Training new model...")

    clf = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=1000)
    )

    clf.fit(X_train, y_train)

    joblib.dump(clf, MODEL_PATH)

    print("Model saved.")


# EVALUATE


y_pred = clf.predict(X_test)

print("\n==============================")
print("SEPARATE TEST SET RESULTS")
print("==============================")

print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["Alert", "Drowsy"]))

X_train shape: (6504, 3)
X_test shape: (2249, 3)
Training new model...
Model saved.

SEPARATE TEST SET RESULTS
Accuracy: 0.5838150289017341

Confusion Matrix:
[[805 401]
 [535 508]]

Classification Report:
              precision    recall  f1-score   support

       Alert       0.60      0.67      0.63      1206
      Drowsy       0.56      0.49      0.52      1043

    accuracy                           0.58      2249
   macro avg       0.58      0.58      0.58      2249
weighted avg       0.58      0.58      0.58      2249



In [ ]:


# 5) TRAIN LOGISTIC REGRESSION


clf = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000)
)

clf.fit(X_train, y_train)


# 6) TEST LOGISTIC REGRESSION


y_pred = clf.predict(X_test)
y_prob = clf.predict_proba(X_test)[:, 1]

print("\n==============================")
print("LOGISTIC REGRESSION RESULTS")
print("==============================")

print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))



# 7) COMPARE WITH PERCLOS RULE


y_pred_perclos = (X_test[:, 0] > 0.8).astype(int)

print("\n==============================")
print("PERCLOS RULE RESULTS")
print("==============================")

print("Accuracy:", accuracy_score(y_test, y_pred_perclos))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_perclos))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_perclos))


LOGISTIC REGRESSION RESULTS
Accuracy: 0.5838150289017341

Confusion Matrix:
[[805 401]
 [535 508]]

Classification Report:
              precision    recall  f1-score   support

           0       0.60      0.67      0.63      1206
           1       0.56      0.49      0.52      1043

    accuracy                           0.58      2249
   macro avg       0.58      0.58      0.58      2249
weighted avg       0.58      0.58      0.58      2249


PERCLOS RULE RESULTS
Accuracy: 0.5326811916407292

Confusion Matrix:
[[1164   42]
 [1009   34]]

Classification Report:
              precision    recall  f1-score   support

           0       0.54      0.97      0.69      1206
           1       0.45      0.03      0.06      1043

    accuracy                           0.53      2249
   macro avg       0.49      0.50      0.37      2249
weighted avg       0.49      0.53      0.40      2249



In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

BOUNDARY = 0.5

def compute_bsa_bsre(y_true, out_scores, boundary=BOUNDARY):
    y_true = np.asarray(y_true).astype(int)
    out_scores = np.asarray(out_scores).astype(float)

    y_pred = (out_scores >= boundary).astype(int)

    Cs = (y_pred != y_true).astype(int)
    bsre = np.mean(Cs * np.abs(out_scores - boundary) ** 2)
    bsa = accuracy_score(y_true, y_pred)

    return bsa, bsre, y_pred


def compute_va_vre(y_true, out_scores, groups, boundary=BOUNDARY):
    y_true = np.asarray(y_true).astype(int)
    out_scores = np.asarray(out_scores).astype(float)
    groups = np.asarray(groups)

    video_true = []
    video_pred = []
    video_errors = []

    for vid in np.unique(groups):
        idx = np.where(groups == vid)[0]

        true_label = int(round(np.mean(y_true[idx])))
        avg_out = np.mean(out_scores[idx])
        pred_label = int(avg_out >= boundary)

        Cv = int(pred_label != true_label)

        video_true.append(true_label)
        video_pred.append(pred_label)
        video_errors.append(Cv * np.abs(avg_out - boundary) ** 2)

    va = accuracy_score(video_true, video_pred)
    vre = np.mean(video_errors)

    return va, vre, np.array(video_true), np.array(video_pred)

In [ ]:
perclos_out = X_test[:, 0]

bsa_p, bsre_p, y_pred_p = compute_bsa_bsre(y_test, perclos_out)
va_p, vre_p, video_true_p, video_pred_p = compute_va_vre(y_test, perclos_out, test_groups)

print("PERCLOS BSA:", bsa_p)
print("PERCLOS BSRE:", bsre_p)
print("PERCLOS VA:", va_p)
print("PERCLOS VRE:", vre_p)

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_p))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_p, target_names=["Alert", "Drowsy"]))

PERCLOS BSA: 0.5335704757670076
PERCLOS BSRE: 0.03400792450965861
PERCLOS VA: 0.5
PERCLOS VRE: 0.018571246927393686

Confusion Matrix:
[[985 221]
 [828 215]]

Classification Report:
              precision    recall  f1-score   support

       Alert       0.54      0.82      0.65      1206
      Drowsy       0.49      0.21      0.29      1043

    accuracy                           0.53      2249
   macro avg       0.52      0.51      0.47      2249
weighted avg       0.52      0.53      0.48      2249



In [ ]:
logreg_out = clf.predict_proba(X_test)[:, 1]
bsa_lr, bsre_lr, y_pred_lr = compute_bsa_bsre(y_test, logreg_out)

va_lr, vre_lr, video_true_lr, video_pred_lr = compute_va_vre(
    y_test,
    logreg_out,
    test_groups
)

print("LOGISTIC REGRESSION BSA:", bsa_lr)
print("LOGISTIC REGRESSION BSRE:", bsre_lr)
print("LOGISTIC REGRESSION VA:", va_lr)
print("LOGISTIC REGRESSION VRE:", vre_lr)

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_lr))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred_lr,
    target_names=["Alert", "Drowsy"]
))

LOGISTIC REGRESSION BSA: 0.5838150289017341
LOGISTIC REGRESSION BSRE: 0.025107293510619022
LOGISTIC REGRESSION VA: 0.7
LOGISTIC REGRESSION VRE: 0.0023432628935934788

Confusion Matrix:
[[805 401]
 [535 508]]

Classification Report:
              precision    recall  f1-score   support

       Alert       0.60      0.67      0.63      1206
      Drowsy       0.56      0.49      0.52      1043

    accuracy                           0.58      2249
   macro avg       0.58      0.58      0.58      2249
weighted avg       0.58      0.58      0.58      2249



In [9]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [11]:
from ultralytics import YOLO

model = YOLO("/content/drive/MyDrive/GP/yolo_experiments/Yolo26/overall_best26.pt")
model2 = YOLO("/content/drive/MyDrive/GP/yolo_experiments/Yolo8/overall_best.pt")

In [11]:
!ls /content/drive/MyDrive/GP/roboflow_dataset/data.yaml

data.yaml  README.roboflow.txt	roboflow.zip  test  train  valid


In [12]:
metrics = model.val(
    data="/content/drive/MyDrive/GP/roboflow_dataset/data.yaml",
    split="test",
    imgsz=640,
    conf=0.25,
    device=0
)

Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Server Edition, 97250MiB)
YOLO26s summary (fused): 122 layers, 9,465,954 parameters, 0 gradients, 20.5 GFLOPs
val: Fast image access ✅ (ping: 0.4±0.1 ms, read: 0.1±0.0 MB/s, size: 89.8 KB)
val: Scanning /content/drive/.shortcut-targets-by-id/1v4vw71crzjzIotC3SRIKu_ujeDJwqWsE/GP/roboflow_dataset/test/labels.cache... 258 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 258/258 63.7Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 17/17 11.6it/s 1.5s
                   all        258        516      0.927      0.924      0.905      0.481
           closed eyes        123        246      0.889      0.877      0.817      0.282
           opened eyes        135        270      0.965       0.97      0.994       0.68
Speed: 0.3ms preprocess, 0.8ms inference, 0.0ms loss, 0.5ms postprocess per image
Results saved to /content/runs/det

In [13]:
metrics = model2.val(
    data="/content/drive/MyDrive/GP/roboflow_dataset/data.yaml",
    split="test",
    imgsz=640,
    conf=0.25,
    device=0
)

Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Server Edition, 97250MiB)
Model summary (fused): 73 layers, 11,126,358 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access ✅ (ping: 0.1±0.0 ms, read: 227.2±92.5 MB/s, size: 91.5 KB)
val: Scanning /content/drive/.shortcut-targets-by-id/1v4vw71crzjzIotC3SRIKu_ujeDJwqWsE/GP/roboflow_dataset/test/labels.cache... 258 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 258/258 135.3Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 17/17 20.0it/s 0.8s
                   all        258        516      0.865      0.905      0.845      0.439
           closed eyes        123        246      0.762      0.809      0.703      0.195
           opened eyes        135        270      0.968          1      0.988      0.682
Speed: 0.3ms preprocess, 0.5ms inference, 0.0ms loss, 0.6ms postprocess per image
Results saved to /content/runs/d

### YOLO Model Comparison Table

Below is a comparison of the key performance metrics (Precision, Recall, mAP50, and mAP50-95) for YOLO26 and YOLO8 models on the test dataset. The metrics are presented for overall performance ('all') and for each specific class ('closed eyes' and 'opened eyes').

In [12]:
print("Evaluating YOLO26 Model (model)...")
yolo26_metrics = model.val(
    data="/content/drive/MyDrive/GP/roboflow_dataset/data.yaml",
    split="test",
    imgsz=640,
    conf=0.25,
    device=0
)
print("YOLO26 Model evaluation complete.")

Evaluating YOLO26 Model (model)...
Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO26s summary (fused): 122 layers, 9,465,954 parameters, 0 gradients, 20.5 GFLOPs


FileNotFoundError: '/content/drive/MyDrive/GP/roboflow_dataset/data.yaml' does not exist

In [15]:
print("Evaluating YOLO8 Model (model2)...")
yolo8_metrics = model2.val(
    data="/content/drive/MyDrive/GP/roboflow_dataset/data.yaml",
    split="test",
    imgsz=640,
    conf=0.25,
    device=0
)
print("YOLO8 Model evaluation complete.")

Evaluating YOLO8 Model (model2)...
Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Server Edition, 97250MiB)
val: Fast image access ✅ (ping: 0.2±0.1 ms, read: 110.5±107.5 MB/s, size: 93.6 KB)
val: Scanning /content/drive/.shortcut-targets-by-id/1v4vw71crzjzIotC3SRIKu_ujeDJwqWsE/GP/roboflow_dataset/test/labels.cache... 258 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 258/258 154.6Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 17/17 19.7it/s 0.9s
                   all        258        516      0.865      0.905      0.845      0.439
           closed eyes        123        246      0.762      0.809      0.703      0.195
           opened eyes        135        270      0.968          1      0.988      0.682
Speed: 0.3ms preprocess, 0.8ms inference, 0.0ms loss, 0.6ms postprocess per image
Results saved to /content/runs/detect/val-4
YOLO8 Model evaluation complete.


In [13]:
import pandas as pd
from ultralytics import YOLO

# Load YOLO26 Model
print("Loading YOLO26 Model...")
model = YOLO("/content/drive/MyDrive/GP/yolo_experiments/Yolo26/overall_best26.pt")
print("YOLO26 Model loaded.")

# Load YOLO8 Model
print("Loading YOLO8 Model...")
model2 = YOLO("/content/drive/MyDrive/GP/yolo_experiments/Yolo8/overall_best.pt")
print("YOLO8 Model loaded.")

# Re-evaluate YOLO26 Model (model) to ensure metrics are fresh and available
print("Evaluating YOLO26 Model (model)...")
yolo26_metrics = model.val(
    data="/content/drive/MyDrive/GP/roboflow_dataset/data.yaml",
    split="test",
    imgsz=640,
    conf=0.25,
    device=0
)
print("YOLO26 Model evaluation complete.")

# Re-evaluate YOLO8 Model (model2)
print("Evaluating YOLO8 Model (model2)...")
yolo8_metrics = model2.val(
    data="/content/drive/MyDrive/GP/roboflow_dataset/data.yaml",
    split="test",
    imgsz=640,
    conf=0.25,
    device=0
)
print("YOLO8 Model evaluation complete.")

def extract_metrics_for_table(metrics_obj, model_name):
    """Extracts relevant metrics from Ultralytics Metrics object for table display."""
    data = []

    # Overall metrics for 'all' class
    data.append({
        "Model": model_name,
        "Class": "all",
        "Precision": f"{metrics_obj.box.mp:.3f}",
        "Recall": f"{metrics_obj.box.mr:.3f}",
        "mAP50": f"{metrics_obj.box.map50:.3f}",
        "mAP50-95": f"{metrics_obj.box.map:.3f}"
    })

    # Per-class metrics
    # Ultralytics metrics.box.p, .r, .ap50, .ap are arrays, indexed by class_id
    for class_id, class_name in metrics_obj.names.items():
        data.append({
            "Model": model_name,
            "Class": class_name,
            "Precision": f"{metrics_obj.box.p[class_id]:.3f}",
            "Recall": f"{metrics_obj.box.r[class_id]:.3f}",
            "mAP50": f"{metrics_obj.box.ap50[class_id]:.3f}",
            "mAP50-95": f"{metrics_obj.box.ap[class_id]:.3f}"
        })
    return data

# Extract metrics for both models
yolo26_table_data = extract_metrics_for_table(yolo26_metrics, "YOLO26")
yolo8_table_data = extract_metrics_for_table(yolo8_metrics, "YOLO8")

# Combine and create DataFrame
df_comparison = pd.DataFrame(yolo26_table_data + yolo8_table_data)
display(df_comparison)

# Convert to LaTeX and print
latex_table = df_comparison.to_latex(index=False, caption='Comparison of YOLO26 and YOLO8 Model Metrics', label='tab:yolo_comparison')
print("\nLaTeX Table:")
print(latex_table)

Loading YOLO26 Model...
YOLO26 Model loaded.
Loading YOLO8 Model...
YOLO8 Model loaded.
Evaluating YOLO26 Model (model)...
Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO26s summary (fused): 122 layers, 9,465,954 parameters, 0 gradients, 20.5 GFLOPs
val: Fast image access ✅ (ping: 1.0±0.9 ms, read: 0.1±0.0 MB/s, size: 90.5 KB)
val: Scanning /content/drive/.shortcut-targets-by-id/1v4vw71crzjzIotC3SRIKu_ujeDJwqWsE/GP/roboflow_dataset/test/labels.cache... 258 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 258/258 47.0Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 17/17 15.0s/it 4:16
                   all        258        516      0.927      0.923      0.905      0.481
           closed eyes        123        246      0.889      0.876      0.816      0.282
           opened eyes        135        270      0.966       0.97      0.994       0.68
Speed: 0.8ms preprocess, 8.3m

,Model,Class,Precision,Recall,mAP50,mAP50-95
0,YOLO26,all,0.927,0.923,0.905,0.481
1,YOLO26,closed eyes,0.889,0.876,0.816,0.282
2,YOLO26,opened eyes,0.966,0.970,0.994,0.680
3,YOLO8,all,0.865,0.905,0.846,0.438
4,YOLO8,closed eyes,0.763,0.809,0.703,0.194
5,YOLO8,opened eyes,0.968,1.000,0.988,0.682



LaTeX Table:
\begin{table}
\caption{Comparison of YOLO26 and YOLO8 Model Metrics}
\label{tab:yolo_comparison}
\begin{tabular}{llllll}
\toprule
Model & Class & Precision & Recall & mAP50 & mAP50-95 \\
\midrule
YOLO26 & all & 0.927 & 0.923 & 0.905 & 0.481 \\
YOLO26 & closed eyes & 0.889 & 0.876 & 0.816 & 0.282 \\
YOLO26 & opened eyes & 0.966 & 0.970 & 0.994 & 0.680 \\
YOLO8 & all & 0.865 & 0.905 & 0.846 & 0.438 \\
YOLO8 & closed eyes & 0.763 & 0.809 & 0.703 & 0.194 \\
YOLO8 & opened eyes & 0.968 & 1.000 & 0.988 & 0.682 \\
\bottomrule
\end{tabular}
\end{table}



In [14]:
latex_table = df_comparison.to_latex(index=False, caption='Comparison of YOLO26 and YOLO8 Model Metrics', label='tab:yolo_comparison')
print(latex_table)

\begin{table}
\caption{Comparison of YOLO26 and YOLO8 Model Metrics}
\label{tab:yolo_comparison}
\begin{tabular}{llllll}
\toprule
Model & Class & Precision & Recall & mAP50 & mAP50-95 \\
\midrule
YOLO26 & all & 0.927 & 0.923 & 0.905 & 0.481 \\
YOLO26 & closed eyes & 0.889 & 0.876 & 0.816 & 0.282 \\
YOLO26 & opened eyes & 0.966 & 0.970 & 0.994 & 0.680 \\
YOLO8 & all & 0.865 & 0.905 & 0.846 & 0.438 \\
YOLO8 & closed eyes & 0.763 & 0.809 & 0.703 & 0.194 \\
YOLO8 & opened eyes & 0.968 & 1.000 & 0.988 & 0.682 \\
\bottomrule
\end{tabular}
\end{table}

